# 论文 20：神经图灵机
## Alex Graves, Greg Wayne, Ivo Danihelka（2014）

### 支持可微读写的外部记忆

NTM 使用外部记忆增强神经网络，并通过注意力机制对这块记忆进行读写。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

## 外部记忆矩阵

In [ ]:
class Memory:
    def __init__(self, num_slots, slot_size):
        """外部存储库
        
        num_slots：存储位置数 (N)
        slot_size：每个记忆向量的大小（M）"""
        self.num_slots = num_slots
        self.slot_size = slot_size
        
        # 将记忆初始化为小的随机值
        self.memory = np.random.randn(num_slots, slot_size) * 0.01
    
    def read(self, weights):
        """使用注意力权重从记忆中读取
        
        weights：（num_slots，）注意力分布
        返回：（slot_size，）记忆行的加权组合"""
        return np.dot(weights, self.memory)
    
    def write(self, weights, erase_vector, add_vector):
        """使用擦除和添加操作写入记忆
        
        weights:(num_slots,)写在哪里
        erase_vector：（slot_size，）要擦除的内容
        add_vector：（slot_size，）添加什么"""
        # 擦除：M_t = M_{t-1} * (1 - w_t ⊗ e_t)
        erase = np.outer(weights, erase_vector)
        self.memory = self.memory * (1 - erase)
        
        # 添加：M_t = M_t + w_t ⊗ a_t
        add = np.outer(weights, add_vector)
        self.memory = self.memory + add
    
    def get_memory(self):
        return self.memory.copy()

# 测试记忆力
memory = Memory(num_slots=8, slot_size=4)
print(f"Memory initialized: {memory.num_slots} slots × {memory.slot_size} dimensions")
print(f"Memory shape: {memory.memory.shape}")

## 基于内容的寻址

根据内容相似性关注记忆位置

In [ ]:
def cosine_similarity(u, v):
    '向量之间的余弦相似度'
    return np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v) + 1e-8)

def softmax(x, beta=1.0):
    'Softmax 温度 beta'
    x = beta * x
    exp_x = np.exp(x - np.max(x))
    return exp_x / np.sum(exp_x)

def content_addressing(memory, key, beta):
    """基于内容的寻址
    
    memory：形状为 (num_slots, slot_size) 的记忆矩阵
    key：（slot_size，）查询向量
    beta：清晰度参数（> 0）
    
    返回：（num_slots，）注意力权重"""
    # 计算每个记忆行的余弦相似度
    similarities = np.array([
        cosine_similarity(key, memory[i]) 
        for i in range(len(memory))
    ])
    
    # 应用具有清晰度的 softmax
    weights = softmax(similarities, beta=beta)
    
    return weights

# 测试内容寻址
key = np.random.randn(memory.slot_size)
beta = 2.0

weights = content_addressing(memory.memory, key, beta)
print(f"\nContent-based addressing:")
print(f"Key shape: {key.shape}")
print(f"Attention weights: {weights}")
print(f"Sum of weights: {weights.sum():.4f}")

# 可视化
plt.figure(figsize=(10, 4))
plt.bar(range(len(weights)), weights)
plt.xlabel('Memory Slot')
plt.ylabel('Attention Weight')
plt.title('Content-Based Addressing Weights')
plt.show()

## 基于位置的寻址

根据相对位置移动注意力权重，以支持顺序访问。

In [ ]:
def interpolation(weights_content, weights_prev, g):
    """在内容和之前的权重之间进行插值
    
    g: [0, 1] 中的门
      g=1：仅使用内容权重
      g=0：仅使用之前的权重"""
    return g * weights_content + (1 - g) * weights_prev

def convolutional_shift(weights, shift_weights):
    """通过轮班分布轮换注意力权重
    
    shift_weights：分布在 [-1, 0, +1] 移位上"""
    num_slots = len(weights)
    shifted = np.zeros_like(weights)
    
    # 应用每个班次
    for shift_idx, shift_amount in enumerate([-1, 0, 1]):
        rolled = np.roll(weights, shift_amount)
        shifted += shift_weights[shift_idx] * rolled
    
    return shifted

def sharpening(weights, gamma):
    """锐化注意力分布
    
    gamma >= 1：值越大=分布越清晰"""
    weights = weights ** gamma
    return weights / (np.sum(weights) + 1e-8)

# 测试基于位置的操作
weights_prev = np.array([0.05, 0.1, 0.2, 0.3, 0.2, 0.1, 0.04, 0.01])
weights_content = content_addressing(memory.memory, key, beta=2.0)

# 插值法
g = 0.7  # 喜欢的内容
weights_gated = interpolation(weights_content, weights_prev, g)

# 转移
shift_weights = np.array([0.1, 0.8, 0.1])  # 大部分停留，很少转移
weights_shifted = convolutional_shift(weights_gated, shift_weights)

# 锐化
gamma = 2.0
weights_sharp = sharpening(weights_shifted, gamma)

# 可视化寻址管道
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

axes[0, 0].bar(range(len(weights_prev)), weights_prev)
axes[0, 0].set_title('Previous Weights')
axes[0, 0].set_ylim(0, 0.5)

axes[0, 1].bar(range(len(weights_content)), weights_content)
axes[0, 1].set_title('Content Weights')
axes[0, 1].set_ylim(0, 0.5)

axes[0, 2].bar(range(len(weights_gated)), weights_gated)
axes[0, 2].set_title(f'Gated (g={g})')
axes[0, 2].set_ylim(0, 0.5)

axes[1, 0].bar(range(len(shift_weights)), shift_weights, color='orange')
axes[1, 0].set_title('Shift Distribution')
axes[1, 0].set_xticks([0, 1, 2])
axes[1, 0].set_xticklabels(['-1', '0', '+1'])

axes[1, 1].bar(range(len(weights_shifted)), weights_shifted, color='green')
axes[1, 1].set_title('After Shift')
axes[1, 1].set_ylim(0, 0.5)

axes[1, 2].bar(range(len(weights_sharp)), weights_sharp, color='red')
axes[1, 2].set_title(f'Sharpened (γ={gamma})')
axes[1, 2].set_ylim(0, 0.5)

plt.tight_layout()
plt.show()

print(f"\nAddressing pipeline complete!")

## 完整的 NTM 头（读/写）

In [ ]:
class NTMHead:
    def __init__(self, memory_slots, memory_size, controller_size):
        self.memory_slots = memory_slots
        self.memory_size = memory_size
        
        # 控制器产生的参数
        # 内容寻址的关键
        self.W_key = np.random.randn(memory_size, controller_size) * 0.1
        
        # 强度（测试版）
        self.W_beta = np.random.randn(1, controller_size) * 0.1
        
        # 门（克）
        self.W_g = np.random.randn(1, controller_size) * 0.1
        
        # 移位权重
        self.W_shift = np.random.randn(3, controller_size) * 0.1
        
        # 锐化（伽玛）
        self.W_gamma = np.random.randn(1, controller_size) * 0.1
        
        # 对于写头：擦除和添加向量
        self.W_erase = np.random.randn(memory_size, controller_size) * 0.1
        self.W_add = np.random.randn(memory_size, controller_size) * 0.1
        
        # 之前的体重
        self.weights_prev = np.ones(memory_slots) / memory_slots
    
    def address(self, memory, controller_output):
        '根据控制器输出计算寻址权重'
        # 内容寻址
        key = np.tanh(np.dot(self.W_key, controller_output))
        beta = np.exp(np.dot(self.W_beta, controller_output))[0] + 1e-4
        weights_content = content_addressing(memory, key, beta)
        
        # 插值法
        g = 1 / (1 + np.exp(-np.dot(self.W_g, controller_output)))[0]  # Sigmoid
        weights_gated = interpolation(weights_content, self.weights_prev, g)
        
        # 转移
        shift_logits = np.dot(self.W_shift, controller_output)
        shift_weights = softmax(shift_logits)
        weights_shifted = convolutional_shift(weights_gated, shift_weights)
        
        # 锐化
        gamma = np.exp(np.dot(self.W_gamma, controller_output))[0] + 1.0
        weights = sharpening(weights_shifted, gamma)
        
        self.weights_prev = weights
        return weights
    
    def read(self, memory, weights):
        '从记忆中读取'
        return memory.read(weights)
    
    def write(self, memory, weights, controller_output):
        '写入记忆'
        erase = 1 / (1 + np.exp(-np.dot(self.W_erase, controller_output)))  # Sigmoid
        add = np.tanh(np.dot(self.W_add, controller_output))
        memory.write(weights, erase, add)

print("NTM Head created with full addressing mechanism")

## 测试任务：复制序列

经典 NTM 任务：将序列从输入复制到输出

In [ ]:
# 简单的复制任务
memory = Memory(num_slots=8, slot_size=4)
controller_size = 16
head = NTMHead(memory.num_slots, memory.slot_size, controller_size)

# 输入顺序
sequence = [
    np.array([1, 0, 0, 0]),
    np.array([0, 1, 0, 0]),
    np.array([0, 0, 1, 0]),
    np.array([0, 0, 0, 1]),
]

# 写入阶段：将序列存储到记忆中
memory_states = [memory.get_memory()]
write_weights_history = []

for i, item in enumerate(sequence):
    # 模拟控制器输出（演示随机）
    controller_out = np.random.randn(controller_size)
    
    # 获取写入权重
    weights = head.address(memory.memory, controller_out)
    write_weights_history.append(weights)
    
    # 写入记忆
    head.write(memory, weights, controller_out)
    memory_states.append(memory.get_memory())

# 可视化写入过程
fig, axes = plt.subplots(1, len(sequence) + 1, figsize=(16, 4))

# 初始记忆
axes[0].imshow(memory_states[0], cmap='RdBu', aspect='auto')
axes[0].set_title('Initial Memory')
axes[0].set_ylabel('Memory Slot')
axes[0].set_xlabel('Dimension')

# 每次写入后
for i in range(len(sequence)):
    axes[i+1].imshow(memory_states[i+1], cmap='RdBu', aspect='auto')
    axes[i+1].set_title(f'After Write {i+1}')
    axes[i+1].set_xlabel('Dimension')

plt.tight_layout()
plt.suptitle('Memory Evolution During Write', y=1.05)
plt.show()

# 显示写入头的注意力模式
write_weights = np.array(write_weights_history).T

plt.figure(figsize=(10, 6))
plt.imshow(write_weights, cmap='viridis', aspect='auto')
plt.colorbar(label='Write Weight')
plt.xlabel('Write Step')
plt.ylabel('Memory Slot')
plt.title('Write Attention Patterns')
plt.show()

print(f"\nWrote {len(sequence)} items to memory")

## 要点

### NTM 架构
1. **控制器**：产生控制信号的神经网络（LSTM/FF）
2. **记忆矩阵**：外部记忆（N × M）
3. **读头**：基于注意力读取记忆
4. **写头**：通过基于注意力的擦除与添加操作写入记忆

### 寻址机制：
1. **基于内容**：与记忆内容的相似性
2. **基于位置**：相对移位（顺序访问）
3. **组合**：在内容和位置之间进行插值

### 寻址流程
```
Content Addressing → Interpolation → Shift → Sharpening
```

### 写操作：
- **擦除**：M_t = M_{t-1} ⊙ (1 - w ⊗ e)
- **添加**：M_t = M_t + (w ⊗ a)
- 组合以允许选择性修改

### 能力：
- 复制和回忆序列
- 学习算法（排序、复制等）
- 泛化到更长的序列
- 可微分记忆访问

### 限制：
- 计算成本较高，需要对所有记忆位置计算注意力
- 训练困难
- 记忆大小固定

### 影响：
- 启发可微记忆研究
- 推动了可微神经计算机（DNC）、记忆网络等后续研究
- 表明神经网络可以学习算法
- 现代外部存储系统的前身